# Ejercicio 3 — Exploración directa de Parquet con DuckDB

Trabajo individual. Se exploran los archivos publicados de taxis amarillos y verdes descargados inicialmente para 2026. Las rutas admiten nuevos años; los resultados reflejan los archivos locales disponibles al ejecutar el notebook.

**Objetivos:** contar archivos y registros, identificar columnas y tipos, obtener muestras y revisar calidad de datos. No se materializan los viajes en tablas ni se cargan completos en Pandas.

Ejecute todas las celdas desde JupyterLab. Los SQL están versionados en `sql/exploracion/`; los resultados completos se guardan como CSV en `docs/resultados_ejercicio_3/`.

In [1]:
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import display

# Funciona al abrir el notebook desde /workspace/notebooks o desde la raíz.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "scripts" / "explore_parquet.py").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "scripts"))
from explore_parquet import create_connection, run_query, write_report, SQL_DIR

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 32)
connection = create_connection()
frames = {}
print("Proyecto:", ROOT)
print("SQL de las vistas externas:\n")
print((SQL_DIR / "00_fuentes.sql").read_text())

Proyecto: /workspace
SQL de las vistas externas:

-- Vistas en memoria: los viajes permanecen en los archivos Parquet.
-- union_by_name admite columnas añadidas entre meses.
CREATE OR REPLACE VIEW yellow_raw AS
SELECT * FROM read_parquet('data/raw/yellow/*/*.parquet', union_by_name = true, filename = true);
CREATE OR REPLACE VIEW green_raw AS
SELECT * FROM read_parquet('data/raw/green/*/*.parquet', union_by_name = true, filename = true);

-- Solo se armonizan nombres; no se eliminan ni imputan registros.
CREATE OR REPLACE VIEW viajes AS
SELECT 'yellow' AS tipo_taxi, filename AS archivo,
       CAST(try_strptime(regexp_extract(filename, '_([0-9]{4}-[0-9]{2})\.parquet$', 1), '%Y-%m') AS DATE) AS mes_archivo,
       VendorID AS vendor_id, tpep_pickup_datetime AS recogida,
       tpep_dropoff_datetime AS llegada, passenger_count, trip_distance,
       RatecodeID AS ratecode_id, store_and_fwd_flag,
       PULocationID AS zona_origen, DOLocationID AS zona_destino,
       payment_type, fare_a

## Fuentes y transformaciones

Las vistas `yellow_raw` y `green_raw` leen directamente Parquet mediante `read_parquet(..., union_by_name=true, filename=true)`. `viajes` armoniza los nombres de fechas, añade tipo de taxi y mes del archivo, y conserva el archivo de origen.

`UNION ALL` conserva todos los registros. No se corrigen fechas, eliminan valores extremos, imputan nulos ni deduplican viajes. `request_source` no existe en los archivos anteriores a junio de 2026; `union_by_name` representa su ausencia con NULL. Los campos exclusivos de un tipo de taxi también reciben NULL para el otro tipo. Las fechas de viaje no se convierten a otra zona horaria.

Todos los archivos utilizados quedan enumerados en el inventario de la primera consulta.

## 01_inventario: Archivos y registros

**Objetivo:** Contar los archivos y registros disponibles por tipo y mes.

**Fuente:** todos los Parquet de `data/raw/yellow/*/*.parquet` y `data/raw/green/*/*.parquet`, enumerados en el inventario. Las consultas de esquema leen sus metadatos; las demás utilizan vistas externas.

**SQL:** `sql/exploracion/01_inventario.sql`.

In [2]:
name = '01_inventario'
print((SQL_DIR / (name + ".sql")).read_text())
frames[name] = run_query(connection, name)
display(frames[name])
display(frames[name].groupby("tipo_taxi").agg(archivos=("archivo", "nunique"), registros=("registros", "sum")))
print("Total de registros:", int(frames[name]["registros"].sum()))

-- Cuenta registros y archivos sin materializar los viajes.
SELECT tipo_taxi, mes_archivo, archivo, count(*) AS registros
FROM viajes
GROUP BY tipo_taxi, mes_archivo, archivo
ORDER BY tipo_taxi, mes_archivo, archivo;



,tipo_taxi,mes_archivo,archivo,registros
0,green,2026-01-01,data/raw/green/2026/green_tripdata_2026-01.par...,40272
1,green,2026-02-01,data/raw/green/2026/green_tripdata_2026-02.par...,37373
2,green,2026-03-01,data/raw/green/2026/green_tripdata_2026-03.par...,44208
3,green,2026-04-01,data/raw/green/2026/green_tripdata_2026-04.par...,44238
4,green,2026-05-01,data/raw/green/2026/green_tripdata_2026-05.par...,44921
5,green,2026-06-01,data/raw/green/2026/green_tripdata_2026-06.par...,44163
6,green,2026-07-01,data/raw/green/2026/green_tripdata_2026-07.par...,41252
7,green,2026-08-01,data/raw/green/2026/green_tripdata_2026-08.par...,40687
8,yellow,2026-01-01,data/raw/yellow/2026/yellow_tripdata_2026-01.p...,3724889
9,yellow,2026-02-01,data/raw/yellow/2026/yellow_tripdata_2026-02.p...,3399866


,archivos,registros
tipo_taxi,,
green,8,337114
yellow,8,29703355


Total de registros: 30040469


**Decisión e interpretación inicial:** Usar el mes del nombre del archivo para inventariar; no confundirlo con la fecha registrada de cada viaje.

El resultado completo queda en `docs/resultados_ejercicio_3/01_inventario.csv`.

## 02_esquemas: Columnas y tipos

**Objetivo:** Identificar columnas, tipos físicos y tipos DuckDB en cada archivo.

**Fuente:** todos los Parquet de `data/raw/yellow/*/*.parquet` y `data/raw/green/*/*.parquet`, enumerados en el inventario. Las consultas de esquema leen sus metadatos; las demás utilizan vistas externas.

**SQL:** `sql/exploracion/02_esquemas.sql`.

In [3]:
name = '02_esquemas'
print((SQL_DIR / (name + ".sql")).read_text())
frames[name] = run_query(connection, name)
display(frames[name])
# Resumen legible; el CSV conserva los campos de cada archivo.
display(frames[name].groupby(["columna", "tipo_duckdb"]).agg(archivos=("archivo", "nunique")).reset_index())

-- Tipos físicos Parquet y tipos interpretados por DuckDB, por archivo.
SELECT file_name AS archivo, name AS columna, type AS tipo_parquet,
       duckdb_type AS tipo_duckdb, repetition_type AS repeticion
FROM parquet_schema(['data/raw/yellow/*/*.parquet', 'data/raw/green/*/*.parquet'])
WHERE num_children IS NULL
ORDER BY archivo, column_id;



,archivo,columna,tipo_parquet,tipo_duckdb,repeticion
0,data/raw/green/2026/green_tripdata_2026-01.par...,VendorID,INT32,INTEGER,OPTIONAL
1,data/raw/green/2026/green_tripdata_2026-01.par...,lpep_pickup_datetime,INT64,TIMESTAMP,OPTIONAL
2,data/raw/green/2026/green_tripdata_2026-01.par...,lpep_dropoff_datetime,INT64,TIMESTAMP,OPTIONAL
3,data/raw/green/2026/green_tripdata_2026-01.par...,store_and_fwd_flag,BYTE_ARRAY,VARCHAR,OPTIONAL
4,data/raw/green/2026/green_tripdata_2026-01.par...,RatecodeID,INT64,BIGINT,OPTIONAL
...,...,...,...,...,...
329,data/raw/yellow/2026/yellow_tripdata_2026-08.p...,total_amount,DOUBLE,DOUBLE,OPTIONAL
330,data/raw/yellow/2026/yellow_tripdata_2026-08.p...,congestion_surcharge,DOUBLE,DOUBLE,OPTIONAL
331,data/raw/yellow/2026/yellow_tripdata_2026-08.p...,Airport_fee,DOUBLE,DOUBLE,OPTIONAL
332,data/raw/yellow/2026/yellow_tripdata_2026-08.p...,cbd_congestion_fee,DOUBLE,DOUBLE,OPTIONAL


,columna,tipo_duckdb,archivos
0,Airport_fee,DOUBLE,8
1,DOLocationID,INTEGER,16
2,PULocationID,INTEGER,16
3,RatecodeID,BIGINT,16
4,VendorID,INTEGER,16
5,cbd_congestion_fee,DOUBLE,16
6,congestion_surcharge,DOUBLE,16
7,ehail_fee,DOUBLE,8
8,extra,DOUBLE,16
9,fare_amount,DOUBLE,16


**Decisión e interpretación inicial:** Conservar los nombres originales en las vistas raw; revisar los esquemas antes de combinar archivos.

El resultado completo queda en `docs/resultados_ejercicio_3/02_esquemas.csv`.

## 03_cambios_esquema: Evolución del esquema

**Objetivo:** Comparar nombres y tipos de columnas entre meses.

**Fuente:** todos los Parquet de `data/raw/yellow/*/*.parquet` y `data/raw/green/*/*.parquet`, enumerados en el inventario. Las consultas de esquema leen sus metadatos; las demás utilizan vistas externas.

**SQL:** `sql/exploracion/03_cambios_esquema.sql`.

In [4]:
name = '03_cambios_esquema'
print((SQL_DIR / (name + ".sql")).read_text())
frames[name] = run_query(connection, name)
display(frames[name])

-- Compara los campos y tipos de cada archivo; no solo la cantidad de columnas.
WITH esquema AS (
    SELECT file_name, name, duckdb_type
    FROM parquet_schema(['data/raw/yellow/*/*.parquet', 'data/raw/green/*/*.parquet'])
    WHERE num_children IS NULL
)
SELECT regexp_extract(file_name, '(yellow|green)_tripdata_', 1) AS tipo_taxi,
       regexp_extract(file_name, '_([0-9]{4}-[0-9]{2})\.parquet$', 1) AS mes,
       count(*) AS columnas,
       string_agg(name || ':' || duckdb_type, ', ' ORDER BY name) AS firma_esquema
FROM esquema
GROUP BY file_name
ORDER BY tipo_taxi, mes;



,tipo_taxi,mes,columnas,firma_esquema
0,green,2026-01,21,"DOLocationID:INTEGER, PULocationID:INTEGER, Ra..."
1,green,2026-02,21,"DOLocationID:INTEGER, PULocationID:INTEGER, Ra..."
2,green,2026-03,21,"DOLocationID:INTEGER, PULocationID:INTEGER, Ra..."
3,green,2026-04,21,"DOLocationID:INTEGER, PULocationID:INTEGER, Ra..."
4,green,2026-05,21,"DOLocationID:INTEGER, PULocationID:INTEGER, Ra..."
5,green,2026-06,22,"DOLocationID:INTEGER, PULocationID:INTEGER, Ra..."
6,green,2026-07,22,"DOLocationID:INTEGER, PULocationID:INTEGER, Ra..."
7,green,2026-08,22,"DOLocationID:INTEGER, PULocationID:INTEGER, Ra..."
8,yellow,2026-01,20,"Airport_fee:DOUBLE, DOLocationID:INTEGER, PULo..."
9,yellow,2026-02,20,"Airport_fee:DOUBLE, DOLocationID:INTEGER, PULo..."


**Decisión e interpretación inicial:** Usar union_by_name y distinguir un campo ausente de un dato no reportado en un campo existente.

El resultado completo queda en `docs/resultados_ejercicio_3/03_cambios_esquema.csv`.

## 04_muestras: Muestra de registros

**Objetivo:** Inspeccionar cinco registros de cada tipo de taxi.

**Fuente:** todos los Parquet de `data/raw/yellow/*/*.parquet` y `data/raw/green/*/*.parquet`, enumerados en el inventario. Las consultas de esquema leen sus metadatos; las demás utilizan vistas externas.

**SQL:** `sql/exploracion/04_muestras.sql`.

In [5]:
name = '04_muestras'
print((SQL_DIR / (name + ".sql")).read_text())
frames[name] = run_query(connection, name)
display(frames[name])

-- Cinco filas por tipo; muestra de inspección, no muestra estadística aleatoria.
(SELECT * FROM viajes WHERE tipo_taxi = 'yellow' LIMIT 5)
UNION ALL
(SELECT * FROM viajes WHERE tipo_taxi = 'green' LIMIT 5);



,tipo_taxi,archivo,mes_archivo,vendor_id,recogida,llegada,passenger_count,trip_distance,ratecode_id,store_and_fwd_flag,zona_origen,zona_destino,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee,request_source,ehail_fee,trip_type
0,yellow,data/raw/yellow/2026/yellow_tripdata_2026-01.p...,2026-01-01,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1,0.97,1,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.50,0.0,0.00,None,NaN,<NA>
1,yellow,data/raw/yellow/2026/yellow_tripdata_2026-01.p...,2026-01-01,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0,0.90,1,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.50,0.0,0.75,None,NaN,<NA>
2,yellow,data/raw/yellow/2026/yellow_tripdata_2026-01.p...,2026-01-01,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0,1.40,1,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.50,0.0,0.75,None,NaN,<NA>
3,yellow,data/raw/yellow/2026/yellow_tripdata_2026-01.p...,2026-01-01,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4,5.58,1,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.50,0.0,0.75,None,NaN,<NA>
4,yellow,data/raw/yellow/2026/yellow_tripdata_2026-01.p...,2026-01-01,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0,2.16,1,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.50,0.0,0.75,None,NaN,<NA>
5,green,data/raw/green/2026/green_tripdata_2026-01.par...,2026-01-01,1,2026-01-01 00:27:58,2026-01-01 00:55:16,2,6.20,1,N,65,233,1,31.7,4.50,1.5,7.50,0.0,1.0,45.20,2.75,NaN,0.75,None,NaN,1
6,green,data/raw/green/2026/green_tripdata_2026-01.par...,2026-01-01,2,2026-01-01 00:44:33,2026-01-01 01:32:56,5,5.36,5,N,66,188,1,50.0,0.00,0.0,10.20,0.0,1.0,61.20,0.00,NaN,0.00,None,NaN,2
7,green,data/raw/green/2026/green_tripdata_2026-01.par...,2026-01-01,1,2026-01-01 00:23:45,2026-01-01 00:45:03,4,10.60,1,N,65,179,1,41.5,1.00,1.5,2.00,0.0,1.0,46.00,0.00,NaN,0.00,None,NaN,1
8,green,data/raw/green/2026/green_tripdata_2026-01.par...,2026-01-01,1,2026-01-01 00:44:33,2026-01-01 01:00:45,1,4.20,1,N,42,141,2,19.8,3.75,1.5,0.00,0.0,1.0,25.05,2.75,NaN,0.00,None,NaN,1
9,green,data/raw/green/2026/green_tripdata_2026-01.par...,2026-01-01,2,2026-01-01 00:46:04,2026-01-01 01:04:40,1,2.76,1,N,95,82,2,19.1,1.00,0.5,0.00,0.0,1.0,21.60,0.00,NaN,0.00,None,NaN,1


**Decisión e interpretación inicial:** La muestra LIMIT sirve para inspeccionar filas, pero no permite estimar la distribución de todos los viajes.

El resultado completo queda en `docs/resultados_ejercicio_3/04_muestras.csv`.

## 05_nulos: Valores nulos

**Objetivo:** Medir valores faltantes en los campos compartidos por ambos tipos.

**Fuente:** todos los Parquet de `data/raw/yellow/*/*.parquet` y `data/raw/green/*/*.parquet`, enumerados en el inventario. Las consultas de esquema leen sus metadatos; las demás utilizan vistas externas.

**SQL:** `sql/exploracion/05_nulos.sql`.

In [6]:
name = '05_nulos'
print((SQL_DIR / (name + ".sql")).read_text())
frames[name] = run_query(connection, name)
display(frames[name])

-- Campos disponibles en ambos taxis; ausencia de columna y valor nulo
-- se distinguen usando 02_esquemas y 03_cambios_esquema.
SELECT tipo_taxi, count(*) AS registros,
       count_if(recogida IS NULL) AS recogida_nula,
       count_if(llegada IS NULL) AS llegada_nula,
       count_if(passenger_count IS NULL) AS pasajeros_nulos,
       count_if(trip_distance IS NULL) AS distancia_nula,
       count_if(total_amount IS NULL) AS total_nulo,
       count_if(payment_type IS NULL) AS pago_nulo,
       count_if(ratecode_id IS NULL) AS tarifa_nula,
       count_if(store_and_fwd_flag IS NULL) AS envio_nulo,
       count_if(zona_origen IS NULL OR zona_destino IS NULL) AS alguna_zona_nula
FROM viajes GROUP BY tipo_taxi ORDER BY tipo_taxi;



,tipo_taxi,registros,recogida_nula,llegada_nula,pasajeros_nulos,distancia_nula,total_nulo,pago_nulo,tarifa_nula,envio_nulo,alguna_zona_nula
0,green,337114,0.0,0.0,48775.0,0.0,0.0,48775.0,48775.0,48775.0,0.0
1,yellow,29703355,0.0,0.0,7716688.0,0.0,0.0,0.0,7716688.0,7716688.0,0.0


**Decisión e interpretación inicial:** Mantener los nulos y analizar qué campos son necesarios para cada pregunta antes de filtrar o imputar.

El resultado completo queda en `docs/resultados_ejercicio_3/05_nulos.csv`.

## 06_calidad: Alertas de calidad

**Objetivo:** Cuantificar fechas discordantes, duraciones, distancias, pasajeros e importes sospechosos.

**Fuente:** todos los Parquet de `data/raw/yellow/*/*.parquet` y `data/raw/green/*/*.parquet`, enumerados en el inventario. Las consultas de esquema leen sus metadatos; las demás utilizan vistas externas.

**SQL:** `sql/exploracion/06_calidad.sql`.

In [7]:
name = '06_calidad'
print((SQL_DIR / (name + ".sql")).read_text())
frames[name] = run_query(connection, name)
display(frames[name])

-- Alertas de exploración: no implican que todos estos registros sean errores.
-- Un viaje >24 h, distancia >100 millas o >6 pasajeros se marca para revisión.
SELECT tipo_taxi, count(*) AS registros,
       min(recogida) AS primera_recogida, max(recogida) AS ultima_recogida,
       count_if(recogida < mes_archivo OR recogida >= mes_archivo + INTERVAL 1 MONTH) AS fecha_fuera_mes_archivo,
       count_if(llegada < recogida) AS duracion_negativa,
       count_if(llegada = recogida) AS duracion_cero,
       count_if(llegada - recogida > INTERVAL 24 HOUR) AS duracion_mayor_24h,
       count_if(trip_distance < 0) AS distancia_negativa,
       count_if(trip_distance = 0) AS distancia_cero,
       count_if(trip_distance > 100) AS distancia_mayor_100_millas,
       count_if(passenger_count <= 0) AS pasajeros_no_positivos,
       count_if(passenger_count > 6) AS pasajeros_mayor_6,
       count_if(fare_amount < 0) AS tarifa_negativa,
       count_if(total_amount < 0) AS total_negativo,
       cou

,tipo_taxi,registros,primera_recogida,ultima_recogida,fecha_fuera_mes_archivo,duracion_negativa,duracion_cero,duracion_mayor_24h,distancia_negativa,distancia_cero,distancia_mayor_100_millas,pasajeros_no_positivos,pasajeros_mayor_6,tarifa_negativa,total_negativo,total_cero,propina_negativa,pago_codigo_no_documentado,tarifa_codigo_no_documentado
0,green,337114,2008-12-31 17:35:31,2026-08-31 23:58:28,98.0,5.0,229.0,4.0,0.0,12212.0,72.0,4527.0,99.0,999.0,1023.0,543.0,69.0,0.0,0.0
1,yellow,29703355,2001-01-01 09:23:58,2026-08-31 23:59:59,146.0,10.0,371673.0,263.0,0.0,952231.0,1223.0,91359.0,28.0,157364.0,161835.0,5258.0,883.0,0.0,0.0


**Decisión e interpretación inicial:** Conservar las alertas; los umbrales de 24 horas, 100 millas y 6 pasajeros son exploratorios y no reglas de eliminación.

El resultado completo queda en `docs/resultados_ejercicio_3/06_calidad.csv`.

## 07_distribuciones: Rangos y percentiles

**Objetivo:** Comparar extremos con percentiles aproximados de los valores originales.

**Fuente:** todos los Parquet de `data/raw/yellow/*/*.parquet` y `data/raw/green/*/*.parquet`, enumerados en el inventario. Las consultas de esquema leen sus metadatos; las demás utilizan vistas externas.

**SQL:** `sql/exploracion/07_distribuciones.sql`.

In [8]:
name = '07_distribuciones'
print((SQL_DIR / (name + ".sql")).read_text())
frames[name] = run_query(connection, name)
display(frames[name])

-- Percentiles aproximados sobre los valores originales, sin limpiar.
WITH valores AS (
    SELECT tipo_taxi, trip_distance, total_amount, passenger_count,
           date_diff('second', recogida, llegada) / 60.0 AS duracion_minutos
    FROM viajes
)
SELECT tipo_taxi,
       min(trip_distance) AS distancia_min, max(trip_distance) AS distancia_max,
       approx_quantile(trip_distance, [0.5, 0.95, 0.99]) AS distancia_p50_p95_p99,
       min(total_amount) AS total_min, max(total_amount) AS total_max,
       approx_quantile(total_amount, [0.5, 0.95, 0.99]) AS total_p50_p95_p99,
       min(duracion_minutos) AS duracion_min, max(duracion_minutos) AS duracion_max,
       approx_quantile(duracion_minutos, [0.5, 0.95, 0.99]) AS duracion_p50_p95_p99,
       min(passenger_count) AS pasajeros_min, max(passenger_count) AS pasajeros_max
FROM valores GROUP BY tipo_taxi ORDER BY tipo_taxi;



,tipo_taxi,distancia_min,distancia_max,distancia_p50_p95_p99,total_min,total_max,total_p50_p95_p99,duracion_min,duracion_max,duracion_p50_p95_p99,pasajeros_min,pasajeros_max
0,green,0.0,179830.92,"[2.066419311969368, 10.525157928029417, 17.780...",-501.5,1678.2,"[20.46185677934834, 57.7190940132087, 97.99630...",-687.000000,2456.366667,"[13.067533089754917, 44.80706624939197, 82.912...",0,9
1,yellow,0.0,328522.20,"[1.8558642635046076, 12.356152069855852, 19.55...",-2560.2,7053.5,"[23.616845997150783, 77.76612682005243, 105.69...",-300517.516667,17165.416667,"[13.924755607669965, 43.76981322484185, 71.295...",0,9


**Decisión e interpretación inicial:** Revisar valores extremos antes de usar promedios o gráficos; los percentiles son aproximados y los nulos se omiten en estas funciones.

El resultado completo queda en `docs/resultados_ejercicio_3/07_distribuciones.csv`.

## 08_categorias_pago: Códigos de pago

**Objetivo:** Examinar pagos, importes negativos y nulos de pasajeros por categoría.

**Fuente:** todos los Parquet de `data/raw/yellow/*/*.parquet` y `data/raw/green/*/*.parquet`, enumerados en el inventario. Las consultas de esquema leen sus metadatos; las demás utilizan vistas externas.

**SQL:** `sql/exploracion/08_categorias_pago.sql`.

In [9]:
name = '08_categorias_pago'
print((SQL_DIR / (name + ".sql")).read_text())
frames[name] = run_query(connection, name)
display(frames[name])

-- Los códigos 0 a 6 están descritos en los diccionarios TLC.
SELECT tipo_taxi, payment_type, count(*) AS registros,
       count_if(total_amount < 0) AS totales_negativos,
       count_if(total_amount = 0) AS totales_cero,
       count_if(passenger_count IS NULL) AS pasajeros_nulos
FROM viajes
GROUP BY tipo_taxi, payment_type ORDER BY tipo_taxi, payment_type;



,tipo_taxi,payment_type,registros,totales_negativos,totales_cero,pasajeros_nulos
0,green,1,219980,137.0,0.0,0.0
1,green,2,65921,43.0,43.0,0.0
2,green,3,1688,564.0,61.0,0.0
3,green,4,750,279.0,71.0,0.0
4,green,<NA>,48775,0.0,368.0,48775.0
5,yellow,0,7716688,94.0,756.0,7716688.0
6,yellow,1,18941008,3511.0,92.0,0.0
7,yellow,2,2708031,39646.0,576.0,0.0
8,yellow,3,98138,17730.0,1296.0,0.0
9,yellow,4,239488,100854.0,2536.0,0.0


**Decisión e interpretación inicial:** El código 0 es Flex Fare según TLC; no clasificarlo como inválido. Un importe negativo requiere contexto y no se elimina automáticamente.

El resultado completo queda en `docs/resultados_ejercicio_3/08_categorias_pago.csv`.

## 09_request_source: Campo agregado entre meses

**Objetivo:** Contar valores de request_source por mes y tipo.

**Fuente:** todos los Parquet de `data/raw/yellow/*/*.parquet` y `data/raw/green/*/*.parquet`, enumerados en el inventario. Las consultas de esquema leen sus metadatos; las demás utilizan vistas externas.

**SQL:** `sql/exploracion/09_request_source.sql`.

In [10]:
name = '09_request_source'
print((SQL_DIR / (name + ".sql")).read_text())
frames[name] = run_query(connection, name)
display(frames[name])

-- Separa presencia estructural del campo y valores efectivamente reportados.
-- NULL en enero-mayo proviene de una columna que no existía.
SELECT tipo_taxi, mes_archivo, request_source, count(*) AS registros
FROM viajes
GROUP BY tipo_taxi, mes_archivo, request_source
ORDER BY tipo_taxi, mes_archivo, request_source;



,tipo_taxi,mes_archivo,request_source,registros
0,green,2026-01-01,NaN,40272
1,green,2026-02-01,NaN,37373
2,green,2026-03-01,NaN,44208
3,green,2026-04-01,NaN,44238
4,green,2026-05-01,NaN,44921
...,...,...,...,...
32,yellow,2026-08-01,EH0004,6253
33,yellow,2026-08-01,EH0010,263
34,yellow,2026-08-01,HV0003,628774
35,yellow,2026-08-01,HV0005,181233


**Decisión e interpretación inicial:** Los NULL anteriores a la incorporación de la columna son ausencia estructural; no inferir demanda a partir de ellos.

El resultado completo queda en `docs/resultados_ejercicio_3/09_request_source.csv`.

## 10_ejemplos_alertas: Ejemplos de alertas

**Objetivo:** Mostrar hasta tres filas por alerta para inspeccionar casos concretos.

**Fuente:** todos los Parquet de `data/raw/yellow/*/*.parquet` y `data/raw/green/*/*.parquet`, enumerados en el inventario. Las consultas de esquema leen sus metadatos; las demás utilizan vistas externas.

**SQL:** `sql/exploracion/10_ejemplos_alertas.sql`.

In [11]:
name = '10_ejemplos_alertas'
print((SQL_DIR / (name + ".sql")).read_text())
frames[name] = run_query(connection, name)
display(frames[name])

-- Hasta tres ejemplos por alerta. Las alertas pueden solaparse.
(SELECT 'fecha_fuera_mes_archivo' AS alerta, tipo_taxi, archivo, recogida,
        llegada, passenger_count, trip_distance, fare_amount, total_amount, payment_type
 FROM viajes
 WHERE recogida < mes_archivo OR recogida >= mes_archivo + INTERVAL 1 MONTH LIMIT 3)
UNION ALL
(SELECT 'duracion_negativa', tipo_taxi, archivo, recogida,
        llegada, passenger_count, trip_distance, fare_amount, total_amount, payment_type
 FROM viajes WHERE llegada < recogida LIMIT 3)
UNION ALL
(SELECT 'distancia_mayor_100_millas', tipo_taxi, archivo, recogida,
        llegada, passenger_count, trip_distance, fare_amount, total_amount, payment_type
 FROM viajes WHERE trip_distance > 100 LIMIT 3)
UNION ALL
(SELECT 'total_negativo', tipo_taxi, archivo, recogida,
        llegada, passenger_count, trip_distance, fare_amount, total_amount, payment_type
 FROM viajes WHERE total_amount < 0 LIMIT 3);



,alerta,tipo_taxi,archivo,recogida,llegada,passenger_count,trip_distance,fare_amount,total_amount,payment_type
0,fecha_fuera_mes_archivo,yellow,data/raw/yellow/2026/yellow_tripdata_2026-01.p...,2025-12-31 23:57:29,2025-12-31 23:57:32,2,0.01,-70.00,-71.50,3
1,fecha_fuera_mes_archivo,yellow,data/raw/yellow/2026/yellow_tripdata_2026-01.p...,2025-12-31 23:57:29,2025-12-31 23:57:32,2,0.01,70.00,71.50,3
2,fecha_fuera_mes_archivo,yellow,data/raw/yellow/2026/yellow_tripdata_2026-01.p...,2025-12-31 23:58:58,2026-01-01 01:04:39,2,40.14,167.20,198.70,2
3,duracion_negativa,yellow,data/raw/yellow/2026/yellow_tripdata_2026-01.p...,2026-01-17 19:30:00,2026-01-17 19:18:18,1,16.50,66.50,67.00,1
4,duracion_negativa,yellow,data/raw/yellow/2026/yellow_tripdata_2026-02.p...,2026-02-22 09:30:00,2026-02-22 09:28:57,1,0.60,16.50,17.00,1
5,duracion_negativa,yellow,data/raw/yellow/2026/yellow_tripdata_2026-02.p...,2026-02-26 04:10:10,2026-02-26 04:10:00,<NA>,0.02,-5.32,-5.82,0
6,distancia_mayor_100_millas,yellow,data/raw/yellow/2026/yellow_tripdata_2026-01.p...,2026-01-02 10:20:45,2026-01-02 10:36:06,2,36610.27,105.27,117.21,2
7,distancia_mayor_100_millas,yellow,data/raw/yellow/2026/yellow_tripdata_2026-01.p...,2026-01-03 17:21:42,2026-01-03 17:33:31,1,100.37,12.80,21.06,1
8,distancia_mayor_100_millas,yellow,data/raw/yellow/2026/yellow_tripdata_2026-01.p...,2026-01-04 13:41:17,2026-01-04 15:58:41,1,112.49,689.70,722.39,2
9,total_negativo,yellow,data/raw/yellow/2026/yellow_tripdata_2026-01.p...,2026-01-01 00:54:05,2026-01-01 00:56:36,1,0.63,-5.10,-10.10,4


**Decisión e interpretación inicial:** Los ejemplos pueden solaparse y no representan una muestra aleatoria; documentar las reglas antes de una limpieza posterior.

El resultado completo queda en `docs/resultados_ejercicio_3/10_ejemplos_alertas.csv`.

## Cómo interpretar las alertas

Las columnas de alertas pueden contar el mismo viaje más de una vez. Los nulos se cuentan por separado y no satisfacen comparaciones numéricas. Los umbrales de 100 millas, 24 horas y 6 pasajeros son señales exploratorias, no límites normativos.

Un pago de tipo 0 es **Flex Fare**, y una tarifa de tipo 99 es **desconocida**, según los diccionarios TLC. No se eliminan por ser códigos desconocidos para nosotros. Los importes negativos se inspeccionan por categoría de pago; su causa no puede determinarse solo con el signo.

Las propinas registradas no incluyen las propinas en efectivo, según los diccionarios de [amarillos](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf) y [verdes](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_green.pdf). La distancia se mide en millas. `request_source` no está explicado en esas versiones del diccionario; se registran sus categorías sin asignarles significado no confirmado.

In [12]:
write_report(connection, frames)
print("Documentación: docs/ejercicio_3_exploracion.md")
print("Resultados completos: docs/resultados_ejercicio_3/")
print("Tablas materializadas:", connection.execute(
    "SELECT count(*) FROM information_schema.tables WHERE table_type = 'BASE TABLE'"
).fetchone()[0])
from IPython.display import Markdown
report = (ROOT / "docs" / "ejercicio_3_exploracion.md").read_text()
interpretation = report.split("### Resultados e interpretación", 1)[1].split("### Valores faltantes", 1)[0]
display(Markdown("## Resultados e interpretación de calidad\n" + interpretation))
connection.close()

Documentación: docs/ejercicio_3_exploracion.md
Resultados completos: docs/resultados_ejercicio_3/
Tablas materializadas: 0


## Resultados e interpretación de calidad


- **yellow:** 7,716,688 registros sin pasajeros (25.98%); 146 recogidas fuera del mes del archivo y 10 duraciones negativas. La primera recogida registrada es 2001-01-01 09:23:58. La distancia máxima es 328,522.20 millas, frente a un percentil 99 aproximado de 19.55 millas. Hay 161,835 importes totales negativos (0.5448%).
- **green:** 48,775 registros sin pasajeros (14.47%); 98 recogidas fuera del mes del archivo y 5 duraciones negativas. La primera recogida registrada es 2008-12-31 17:35:31. La distancia máxima es 179,830.92 millas, frente a un percentil 99 aproximado de 17.78 millas. Hay 1,023 importes totales negativos (0.3035%).

En yellow, todos los pasajeros nulos coinciden con la categoría pago 0 (Flex Fare), y todos los registros de esa categoría tienen pasajeros nulos. En green, todos los pasajeros nulos coinciden con la categoría pago nulo, y todos los registros de esa categoría tienen pasajeros nulos. La consulta 08 permite revisar asociaciones entre nulos y categorías de pago. Una asociación observada no explica su causa: excluir todas las filas con pasajeros nulos puede cambiar la cobertura de modalidades. Por eso no se aplica una limpieza automática.

Los rangos y percentiles deben revisarse antes de calcular promedios. Una recogida fuera del mes nominal necesita inspección, incluyendo posibles viajes alrededor del cambio de año. El cambio de esquema se registra como evolución de la fuente, no como corrupción de archivos.



## 3.9 — Consulta directa de Parquet

DuckDB ejecuta SQL sobre archivos externos sin importarlos primero a una tabla. Las vistas contienen la definición de lectura, no una copia de los viajes. Parquet organiza los datos por columnas y grupos de filas; DuckDB puede aprovechar columnas seleccionadas, filtros y metadatos para reducir lecturas cuando la consulta lo permite.

Esto permite explorar un conjunto grande sin cargarlo completo en un DataFrame. En este notebook solo se convierten a Pandas los metadatos, agregados y muestras pequeñas. Los perfiles de calidad y percentiles sí requieren examinar valores y consumir recursos; no toda consulta se resuelve solamente con metadatos.

Referencia: [documentación oficial de DuckDB sobre Parquet](https://duckdb.org/docs/stable/data/parquet/overview). Las conclusiones de calidad no sustituyen una validación exhaustiva de duplicados o de zonas. El ejercicio 4 desarrollará las preguntas analíticas y los hallazgos del análisis exploratorio.